In [ ]:
"""
E-Commerce Churn Dataset — Full Model Pipeline
Step 1: Train multiple models on E_Comm_Train_Balanced.csv
Step 2: Evaluate all models on E_Comm_Test_Real.csv (real, untouched)
Step 3: Identify best model (by F1-Score)
Step 4: Hyperparameter tune the best model using RandomizedSearchCV
Step 5: Save the final tuned model (only if it improves on baseline)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, AdaBoostClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

pd.set_option('display.width', 140)

# ============================================================
# STEP 0: LOAD TRAIN (BALANCED) AND TEST (REAL) DATA
# ============================================================
train_df = pd.read_csv('E_Comm_Train_Balanced.csv')
test_df = pd.read_csv('E_Comm_Test_Real.csv')

TARGET = 'Churn'
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train class balance: {y_train.value_counts().to_dict()}")
print(f"Test class balance (real): {y_test.value_counts().to_dict()}")

# ============================================================
# STEP 1: DEFINE ALL BASELINE MODELS
# ============================================================
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Extra Trees': ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, random_state=42, eval_metric='logloss', n_jobs=-1),
    'LightGBM': LGBMClassifier(n_estimators=200, random_state=42, verbose=-1),
    'AdaBoost': AdaBoostClassifier(n_estimators=200, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Naive Bayes': GaussianNB(),
    'SVM (RBF)': SVC(probability=True, random_state=42)
}

# ============================================================
# STEP 2: TRAIN + EVALUATE EACH BASELINE MODEL ON REAL TEST SET
# ============================================================
results = []
fitted_models = {}

for name, model in models.items():
    print(f"\n{'='*60}\nTraining: {name}\n{'='*60}")

    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start
    fitted_models[name] = model

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    results.append({
        'Model': name, 'Accuracy': acc, 'Precision': prec,
        'Recall': rec, 'F1-Score': f1, 'ROC-AUC': roc_auc,
        'Train Time (s)': round(train_time, 2)
    })

    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} "
          f"| F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f} | Time: {train_time:.2f}s")

results_df = pd.DataFrame(results).sort_values(by='F1-Score', ascending=False).reset_index(drop=True)
print(f"\n{'='*60}\nBASELINE MODEL COMPARISON (sorted by F1-Score)\n{'='*60}")
print(results_df.to_string(index=False))
results_df.to_csv('model_comparison_results.csv', index=False)
print("\nSaved -> model_comparison_results.csv")

# ============================================================
# STEP 3: PLOTS FOR BASELINE COMPARISON
# ============================================================
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
results_df.set_index('Model')[metrics_to_plot].plot(kind='bar', figsize=(14, 6))
plt.title('Baseline Model Comparison Across Metrics')
plt.ylabel('Score')
plt.xticks(rotation=45, ha='right')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('model_comparison_chart.png', dpi=150)
plt.show()
print("Saved -> model_comparison_chart.png")

top_models = results_df.head(4)['Model'].tolist()

fig, axes = plt.subplots(1, len(top_models), figsize=(6 * len(top_models), 5))
for ax, name in zip(axes, top_models):
    y_pred = fitted_models[name].predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
    ax.set_title(f'{name}\nConfusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrices_top4.png', dpi=150)
plt.show()
print("Saved -> confusion_matrices_top4.png")

plt.figure(figsize=(8, 6))
for name in top_models:
    y_proba = fitted_models[name].predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Top 4 Models')
plt.legend()
plt.tight_layout()
plt.savefig('roc_curves_top4.png', dpi=150)
plt.show()
print("Saved -> roc_curves_top4.png")

# ============================================================
# STEP 4: IDENTIFY BEST BASELINE MODEL
# ============================================================
best_model_name = results_df.iloc[0]['Model']
baseline_f1 = results_df.iloc[0]['F1-Score']
print(f"\n{'='*60}\nBEST BASELINE MODEL: {best_model_name} (F1 = {baseline_f1:.4f})\n{'='*60}")

# Save untuned best model as a fallback checkpoint
joblib.dump(fitted_models[best_model_name], 'best_churn_model_untuned.pkl')
joblib.dump(list(X_train.columns), 'model_feature_columns.pkl')
print("Saved untuned checkpoint -> best_churn_model_untuned.pkl")

# ============================================================
# STEP 5: HYPERPARAMETER GRIDS FOR TUNING
# ============================================================
model_configs = {
    'Logistic Regression': {
        'estimator': LogisticRegression(max_iter=1000, random_state=42),
        'param_grid': {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l1', 'l2'], 'solver': ['liblinear']}
    },
    'Decision Tree': {
        'estimator': DecisionTreeClassifier(random_state=42),
        'param_grid': {'max_depth': [5, 10, 15, 20, None], 'min_samples_split': [2, 5, 10],
                        'min_samples_leaf': [1, 2, 4], 'criterion': ['gini', 'entropy']}
    },
    'Random Forest': {
        'estimator': RandomForestClassifier(random_state=42, n_jobs=-1),
        'param_grid': {'n_estimators': [100, 200, 300, 500], 'max_depth': [10, 20, 30, None],
                        'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4],
                        'max_features': ['sqrt', 'log2']}
    },
    'Extra Trees': {
        'estimator': ExtraTreesClassifier(random_state=42, n_jobs=-1),
        'param_grid': {'n_estimators': [100, 200, 300, 500], 'max_depth': [10, 20, 30, None],
                        'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4],
                        'max_features': ['sqrt', 'log2']}
    },
    'Gradient Boosting': {
        'estimator': GradientBoostingClassifier(random_state=42),
        'param_grid': {'n_estimators': [100, 200, 300], 'learning_rate': [0.01, 0.05, 0.1, 0.2],
                        'max_depth': [3, 4, 5, 6], 'subsample': [0.7, 0.8, 0.9, 1.0]}
    },
    'XGBoost': {
        'estimator': XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1),
        'param_grid': {'n_estimators': [100, 200, 300, 500], 'learning_rate': [0.01, 0.05, 0.1, 0.2],
                        'max_depth': [3, 4, 5, 6, 8], 'subsample': [0.7, 0.8, 0.9, 1.0],
                        'colsample_bytree': [0.7, 0.8, 0.9, 1.0]}
    },
    'LightGBM': {
        'estimator': LGBMClassifier(random_state=42, verbose=-1),
        'param_grid': {'n_estimators': [100, 200, 300, 500], 'learning_rate': [0.01, 0.05, 0.1, 0.2],
                        'max_depth': [3, 5, 7, -1], 'num_leaves': [15, 31, 63, 127],
                        'subsample': [0.7, 0.8, 0.9, 1.0]}
    },
    'AdaBoost': {
        'estimator': AdaBoostClassifier(random_state=42),
        'param_grid': {'n_estimators': [50, 100, 200, 300], 'learning_rate': [0.01, 0.05, 0.1, 0.5, 1.0]}
    },
    'K-Nearest Neighbors': {
        'estimator': KNeighborsClassifier(n_jobs=-1),
        'param_grid': {'n_neighbors': [3, 5, 7, 9, 11, 15], 'weights': ['uniform', 'distance'], 'p': [1, 2]}
    },
    'SVM (RBF)': {
        'estimator': SVC(probability=True, random_state=42),
        'param_grid': {'C': [0.1, 1, 10, 100], 'gamma': ['scale', 'auto', 0.001, 0.01, 0.1], 'kernel': ['rbf']}
    }
}

# ============================================================
# STEP 6: TUNE THE BEST MODEL (skip tuning if Naive Bayes wins — no useful params)
# ============================================================
if best_model_name == 'Naive Bayes':
    print("\nBest model is Naive Bayes — no meaningful hyperparameters to tune.")
    print("Using untuned Naive Bayes as the final model.")
    final_model = fitted_models[best_model_name]
    final_f1 = baseline_f1
    search = None

else:
    config = model_configs[best_model_name]
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    search = RandomizedSearchCV(
        estimator=config['estimator'],
        param_distributions=config['param_grid'],
        n_iter=30,
        scoring='f1',
        cv=cv,
        verbose=2,
        random_state=42,
        n_jobs=-1
    )

    print(f"\nStarting RandomizedSearchCV for {best_model_name}...")
    start = time.time()
    search.fit(X_train, y_train)
    tune_time = time.time() - start

    print(f"\nTuning completed in {tune_time:.1f}s")
    print(f"Best parameters: {search.best_params_}")
    print(f"Best CV F1-Score: {search.best_score_:.4f}")

    tuned_model = search.best_estimator_
    y_pred = tuned_model.predict(X_test)
    y_proba = tuned_model.predict_proba(X_test)[:, 1]

    tuned_f1 = f1_score(y_test, y_pred)

    print(f"\n{'='*60}\nTUNED MODEL PERFORMANCE ({best_model_name})\n{'='*60}")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
    print(f"F1-Score:  {tuned_f1:.4f}")
    print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

    print(f"\nBaseline F1 (untuned): {baseline_f1:.4f}")
    print(f"Tuned F1:              {tuned_f1:.4f}")
    print(f"Improvement:           {tuned_f1 - baseline_f1:+.4f}")

    # Keep whichever version is actually better
    if tuned_f1 > baseline_f1:
        final_model = tuned_model
        final_f1 = tuned_f1
        print("\n✅ Tuning improved performance — using tuned model as final.")
    else:
        final_model = fitted_models[best_model_name]
        final_f1 = baseline_f1
        print("\n⚠️ Tuning did NOT improve F1-Score — keeping untuned model as final.")

# ============================================================
# STEP 7: SAVE THE FINAL MODEL (best baseline OR tuned, whichever wins)
# ============================================================
joblib.dump(final_model, 'best_churn_model.pkl')
joblib.dump(list(X_train.columns), 'model_feature_columns.pkl')

if search is not None:
    joblib.dump(search.best_params_, 'best_model_hyperparameters.pkl')

    tuning_summary = pd.DataFrame([{
        'Model': best_model_name,
        'Baseline F1': baseline_f1,
        'Tuned F1': tuned_f1,
        'Final F1 Used': final_f1,
        'Best Params': str(search.best_params_)
    }])
    tuning_summary.to_csv('tuning_results.csv', index=False)
    print("Saved -> tuning_results.csv")

print(f"\n{'='*60}")
print(f"FINAL MODEL SAVED: {best_model_name}")
print(f"Final F1-Score: {final_f1:.4f}")
print(f"{'='*60}")
print("Saved -> best_churn_model.pkl")
print("Saved -> model_feature_columns.pkl")
print("\nThis 'best_churn_model.pkl' is ready to be loaded into your FastAPI service.")